# **FAKE NEWS DETECTION SYSTEM USING MACHINE LEARNING**

***STEP 1: SET UP THE ENVIRONMENT***

*Install and import all the required libraries*

In [ ]:
# Install required libraries (if not already installed)
!pip install pandas scikit-learn nltk

# Import Python libraries
import pandas as pd
import numpy as np
import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK stopwords
nltk.download('stopwords')
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


***STEP 2: DATA ACQUISITION***

*We'll load the Fake.csv and True.csv files into colab.These files contain the fake and real news articles*

In [ ]:
# Load the datasets
fake_df = pd.read_csv("Fake.csv")
real_df = pd.read_csv("True.csv")

# Add labels: 0 = Fake, 1 = Real
fake_df['label'] = 0
real_df['label'] = 1

# Combine both datasets
data = pd.concat([fake_df, real_df], axis=0)
data = data[['text', 'label']]  # Keep only the 'text' and 'label' columns

# Shuffle the data
data = data.sample(frac=1).reset_index(drop=True)

# Display first few rows
data.head()


,text,label
0,"Last week, the alleged president made some jaw...",0
1,WASHINGTON (Reuters) - President Donald Trump ...,1
2,JAKARTA (Reuters) - Indonesia intends to send ...,1
3,BRASILIA (Reuters) - Brazil s President Michel...,1
4,Judge Sri Srinivasan is a possible pick for Ob...,0


***STEP 3: PREPROCESSING***

*We want to clean the news text data*

In [ ]:
# Function to clean the text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\W', ' ', text)  # Remove all non-word characters
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with single space
    text = re.sub(r'\d', '', text)  # Remove digits
    return text

# Apply the function to the 'text' column
data['text'] = data['text'].apply(clean_text)

# Show the cleaned text
data.head()


,text,label
0,last week the alleged president made some jaw ...,0
1,washington reuters president donald trump unve...,1
2,jakarta reuters indonesia intends to send a di...,1
3,brasilia reuters brazil s president michel tem...,1
4,judge sri srinivasan is a possible pick for ob...,0


***STEP 4: FEATURE SELECTION (VECTORIZATION)***

*Convert the cleaned text into numerical features using TF-IDF Vectorization*

In [ ]:
# Split features and labels
X = data['text']
y = data['label']

# Convert text to numerical data using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
X_vectors = vectorizer.fit_transform(X)

# Show the shape of the vectorized data
print("TF-IDF matrix shape:", X_vectors.shape)


TF-IDF matrix shape: (44898, 114628)


***STEP 5: MODEL BUILDING***

*Train a machine learning model that can classify news as real(1) or fake(0)*

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_vectors, y, test_size=0.2, random_state=42)

# Initialize the Logistic Regression model
model = LogisticRegression()

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)


***STEP 6: TRAINING EVALUATION AND METRICES***

*Evaluate how well the model is performing*

In [ ]:
# Accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.9847

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4594
           1       0.98      0.99      0.98      4386

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980

Confusion Matrix:
[[4518   76]
 [  61 4325]]


***STEP 8: PREPREDICTING NEW NEWS ARTICLES***

In [ ]:
def predict_news(news_text):
    # Clean the text (same cleaning used before)
    cleaned = clean_text(news_text)

    # Convert the cleaned text to TF-IDF vector
    vectorized = vectorizer.transform([cleaned])

    # Predict using the trained model
    prediction = model.predict(vectorized)

    # Interpret the result
    result = "Real News 🟢" if prediction[0] == 1 else "Fake News 🔴"
    return result

# Example usage
sample_news = "The government announces a new policy to support small businesses."
print("Prediction:", predict_news(sample_news))


Prediction: Real News 🟢


In [ ]:
# Example usage
any_news = "A Fargo,North Dakota,man was arrested for clearing snow with a flamethrower."
print("Prediction:", predict_news(any_news))

Prediction: Fake News 🔴


***OR***

***STEP 9: INTERACTIVE NEWS PREDICTION (USING input())***

In [ ]:
# Function to predict whether news is fake or real
def predict_news(news):
    news = clean_text(news)
    vectorized = vectorizer.transform([news])
    prediction = model.predict(vectorized)
    return "Real News 🟢" if prediction[0] == 1 else "Fake News 🔴"


In [ ]:
# Interactive prediction from user input
news_input = input("Enter a news article to check if it's Fake or Real:\n")

# Predict and print result
print("Prediction:", predict_news(news_input))


Enter a news article to check if it's Fake or Real:
The Prime Minister inaugurated the new metro line in the city, improving transportation for thousands.
Prediction: Real News 🟢
